In [1]:
import numpy as np
import random
# =============================================================
#  variables.py — Constantes del juego "Hundir la Flota"
#  Autora : Claudia (Scrum Master)
#  Uso    : Importar en clases.py, funciones.py y main.py
# =============================================================

# ── Dimensiones del tablero ───────────────────────────────────
board_size = 10          # Tablero de 10 x 10 celdas

# ── Flota de barcos ───────────────────────────────────────────
# Cada entrada: nombre -> {"eslora": int, "cantidad": int}
barcos = {
    "fragata":     {"eslora": 1, "cantidad": 4},   # 4 × 1 celda
    "destructor":  {"eslora": 2, "cantidad": 3},   # 3 × 2 celdas
    "crucero":     {"eslora": 3, "cantidad": 2},   # 2 × 3 celdas
    "portaviones": {"eslora": 4, "cantidad": 1},   # 1 × 4 celdas
}

# Número total de "vidas" (celdas con barco) de cada jugador
total_barcos_celdas = sum(
    datos["eslora"] * datos["cantidad"]
    for datos in barcos.values()
)  # → 4*1 + 3*2 + 2*3 + 1*4 = 20

# ── Símbolos del tablero ──────────────────────────────────────
# Estos valores se usan para rellenar los arrays de numpy
symbol_agua      = "~"   # Agua sin disparar
symbol_barco       = "O"   # Barco intacto (tablero propio)
symbol_disparo        = "X"   # Disparo que impactó un barco
symbol_fallo       = "*"   # Disparo que cayó en agua

# Diccionario agrupado (útil para mostrar la leyenda)
symbols = {
    "agua":    symbol_agua,
    "barco":   symbol_barco,
    "impacto": symbol_disparo,
    "fallado": symbol_fallo,
}

# ── Orientaciones para colocar barcos ────────────────────────
orientations = ["H", "V"]   # Horizontal / Vertical

# ── Mensajes del juego ────────────────────────────────────────
msg_bienvenida = """
╔══════════════════════════════════════════╗
║        ⚓  HUNDIR LA FLOTA  ⚓          ║
╚══════════════════════════════════════════╝
¡Bienvenido! Destruye toda la flota enemiga
antes de que la máquina hunda la tuya.
"""

msg_instrucciones = """
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  INSTRUCCIONES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  • El tablero es de 10 x 10 (filas 0-9, columnas 0-9).
  • Introduce coordenadas como: fila columna  (ej. 3 7)
  • Si aciertas, vuelves a disparar.
  • Si fallas, dispara la máquina.
  • Gana quien hunda toda la flota rival.

  FLOTA (cada jugador):
    Fragata     (x4) ── eslora 1
    Destructor  (x3) ── eslora 2
    Crucero     (x2) ── eslora 3
    Portaviones (x1) ── eslora 4

  SIMBOLOGÍA DEL TABLERO:
    ~  Agua sin disparar
    O  Barco intacto (solo tu tablero)
    X  Impacto en barco
    *  Disparo fallado
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

msg_tu_turno       = "\n🎯 Tu turno — introduce coordenadas (fila columna):"
msg_maquina_turno  = "\n🤖 Turno de la máquina..."
msg_impacto        = "💥 ¡IMPACTO!"
msg_agua           = "💧 Agua. Turno para la máquina."
msg_maquina_hit    = "🤖 ¡La máquina ha acertado en ({}, {})!"
msg_maquina_miss   = "🤖 La máquina ha fallado en ({}, {})."
msg_ya_disparado   = "⚠️  Ya disparaste en esas coordenadas. Elige otras."
msg_coord_invalida = "⚠️  Coordenadas inválidas. Usa números entre 0 y 9."
msg_ganas          = "\n🏆 ¡Enhorabuena! Has hundido toda la flota enemiga. ¡GANASTE!"
msg_pierdes        = "\n💀 La máquina ha hundido tu flota. ¡Has perdido!"

# ── Etiquetas de los tableros al imprimir ─────────────────────
label_my_board      = "═══  MI TABLERO  ═══"
label_enemy_board   = "═══  TABLERO ENEMIGO  ═══"

In [ ]:
class Tablero:
    def __init__(self, player_id: str):
        """Inicializa el tablero del jugador."""
        self.player_id = player_id
        self.size = board_size
        self.board = np.full((self.size, self.size), symbol_agua)
        self.tracking = np.full((self.size, self.size), symbol_agua)
        self.vidas = 0

    def colocar_barco(self, tipo_barco, fila, columna, orientacion):
        """
        Coloca un barco en la posición indicada con orientación N, S, E u O.
        Devuelve True si se colocó correctamente, False si no cabe o choca.
        """

        eslora = barcos[tipo_barco]["eslora"]

        # --- 1. Comprobar si cabe dentro del tablero ---
        if orientacion == "N":
            if fila - (eslora - 1) < 0:                   # Para Norte y Sur, la columna no cambia, por eso se comprueba
                return False                                # que la fila esté en el rango eslora - 1
        elif orientacion == "S":                            # En Este y Oeste, lo que se comprueba es que la columna,
            if fila + (eslora - 1) >= self.size:            # sea menor que la matriz (self.size)
                return False
        elif orientacion == "E":
            if columna + (eslora - 1) >= self.size:
                return False
        elif orientacion == "O":
            if columna - (eslora - 1) < 0:
                return False

        # --- 2. Comprobar que no pisa otro barco ---
        posiciones = []

        for i in range(eslora):
            if orientacion == "N":
                f, c = fila - i, columna         # Para N y S, la columna no cambia, solo la fila
            elif orientacion == "S":                # Para E y O, la fila no cambia, solo la columna
                f, c = fila + i, columna
            elif orientacion == "E":
                f, c = fila, columna + i
            elif orientacion == "O":
                f, c = fila, columna - i

            if self.board[f, c] == symbol_barco:
                return False

            posiciones.append((f, c)) # se guardan las posiciones si todas cumplen. Una vez guardadas, coloca el barco

        # --- 3. Colocar el barco ---
        for f, c in posiciones:
            self.board[f, c] = symbol_barco
            self.vidas += eslora
        
        # --- 4. Imprimir el tablero después de colocar ---
        print(self.board)


        return True
    
    def recibir_disparo(self, fila, columna):
    
    casilla = self.board[fila, columna]

    # --- 1. Ya disparado antes ---

    if casilla == symbol_fallo or casilla == symbol_disparo:
        return "Ya has disparado aquí"

    # --- 2. Agua ---
    if casilla == symbol_agua:
        self.board[fila, columna] = symbol_fallo
        return "Agua"

    # --- 3. Barco tocado ---
    if casilla == symbol_barco:
        self.board[fila, columna] = symbol_disparo
        self.vidas -= 1   # restamos una vida
        return "Tocado"
        



   
   
   


In [ ]:
    
    # --- 3. Barco tocado ---
    if casilla == symbol_barco:
        self.board[fila, columna] = symbol_disparo
        self.vidas -= 1   # restamos una vida

        # Comprobamos si ha hundido el barco, o solo lo ha tocado:
        if self.barco_hundido(fila, columna):
            return "Hundido"
        else:
            return "Tocado"
        
    def barco_hundido(self, fila, columna):
    
    # Comprueba si el barco al que pertenece esta casilla está completamente hundido.
    # Busca en las 4 direcciones hasta que encuentre agua o el borde.
    # Si encuentra alguna parte del barco sin disparar, NO está hundido.

    # Direcciones: arriba, abajo, derecha, izquierda
    direcciones = [(-1,0), (1,0), (0,1), (0,-1)]

    for df, dc in direcciones:
        f, c = fila + df, columna + dc

        while 0 <= f < self.size and 0 <= c < self.size:
            if self.board[f, c] == symbol_barco:
                return False  # queda parte sin tocar
            if self.board[f, c] == symbol_agua or self.board[f, c] == symbol_fallo:
                break  # ya no es parte del barco
            f += df
            c += dc

    return True

In [3]:
mi_tablero = Tablero("Marta")
mi_tablero.board

array([['~', '~', '~', '~', '~', '~', '~', '~', '~', '~'],
       ['~', '~', '~', '~', '~', '~', '~', '~', '~', '~'],
       ['~', '~', '~', '~', '~', '~', '~', '~', '~', '~'],
       ['~', '~', '~', '~', '~', '~', '~', '~', '~', '~'],
       ['~', '~', '~', '~', '~', '~', '~', '~', '~', '~'],
       ['~', '~', '~', '~', '~', '~', '~', '~', '~', '~'],
       ['~', '~', '~', '~', '~', '~', '~', '~', '~', '~'],
       ['~', '~', '~', '~', '~', '~', '~', '~', '~', '~'],
       ['~', '~', '~', '~', '~', '~', '~', '~', '~', '~'],
       ['~', '~', '~', '~', '~', '~', '~', '~', '~', '~']], dtype='<U1')

In [8]:
mi_tablero.colocar_barco("destructor", 5, 5, "N")


False

In [7]:
mi_tablero.colocar_barco("portaviones", 1, 2, "E")


[['~' '~' '~' '~' '~' '~' '~' '~' '~' '~']
 ['~' '~' 'O' 'O' 'O' 'O' '~' '~' '~' '~']
 ['~' '~' '~' '~' '~' '~' '~' '~' '~' '~']
 ['~' '~' '~' '~' '~' '~' '~' '~' '~' '~']
 ['~' '~' '~' '~' '~' 'O' '~' '~' '~' '~']
 ['~' '~' '~' '~' '~' 'O' '~' '~' '~' '~']
 ['~' '~' '~' '~' '~' '~' '~' '~' '~' '~']
 ['O' '~' '~' '~' '~' '~' '~' '~' '~' '~']
 ['O' '~' '~' '~' '~' '~' '~' '~' '~' '~']
 ['O' '~' '~' '~' '~' '~' '~' '~' '~' '~']]


True

In [6]:
mi_tablero.colocar_barco("crucero", 9, 0, "N")


[['~' '~' '~' '~' '~' '~' '~' '~' '~' '~']
 ['~' '~' '~' '~' '~' '~' '~' '~' '~' '~']
 ['~' '~' '~' '~' '~' '~' '~' '~' '~' '~']
 ['~' '~' '~' '~' '~' '~' '~' '~' '~' '~']
 ['~' '~' '~' '~' '~' 'O' '~' '~' '~' '~']
 ['~' '~' '~' '~' '~' 'O' '~' '~' '~' '~']
 ['~' '~' '~' '~' '~' '~' '~' '~' '~' '~']
 ['O' '~' '~' '~' '~' '~' '~' '~' '~' '~']
 ['O' '~' '~' '~' '~' '~' '~' '~' '~' '~']
 ['O' '~' '~' '~' '~' '~' '~' '~' '~' '~']]


True